Build a compact CNN that ingests the 1-channel FFT log-magnitude image and outputs (1) a binary logit (real vs AI) and (2) a 128-D embedding for future fusion.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GDRIVE_DATA_DIR = "/content/drive/MyDrive/datasets"

In [ ]:
import os
import time
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np
from PIL import Image

# Where to save best checkpoints & logs
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
data_dir = f"{GDRIVE_DATA_DIR}/OpenFake"
epochs = 30
batch_size = 64
lr = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"

# Early Stopping parameters
PATIENCE = 3
patience_counter = 0

In [ ]:
# Steps 1 - 4
class FFTDataset(Dataset):
    def __init__(self, img_paths: list[str], labels: list[int], img_size=256):
        """
        img_paths (list[str]): image paths
        labels (list[int]): labels for 0 = real, 1 = fake
        """
        self.img_paths = img_paths
        self.labels = labels
        self.resize = transforms.Resize((img_size, img_size))
        self.to_tensor = transforms.ToTensor()
        self.to_gray = transforms.Grayscale()


    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # get 3 channel image
        img = Image.open(self.img_paths[idx]).convert('RGB')

        # resize
        img = self.resize(img)

        # RBG -> grayscale
        img_gray = transforms.functional.rgb_to_grayscale(img)

        # PIL -> Tensor
        img_gray = transforms.functional.to_tensor(img_gray)

        # spatial -> frequency
        fft = torch.fft.fft2(img_gray)

        # move 0-frequency component to center of image
        fft_shift = torch.fft.fftshift(fft)

        # log (|1 + FFT|)
        fft_log_mag = torch.log1p(torch.abs(fft_shift))

        fft_log_mag = self.resize(fft_log_mag)

        # fft log-mag image as float 32 tensory, label
        return fft_log_mag.float(), torch.tensor(self.labels[idx], dtype=torch.float32)

In [ ]:
# CNN Model Definition
class CompactFFTNet(nn.Module):
    def __init__(self, input_channels=1, depth=3, base_filters=16, dropout=0.2, embedding_dim=128):
        """
        input_channels = 1 because grayscale
        depth : num convolutional blcoks
        base_filters: num filters in first convolution layer
        dropout: dropout rate to prevent overfitting
        embedding_dim: feature embedding vector
        """
        super().__init__()

        self.input_norm = nn.BatchNorm2d(input_channels)

        self.channel_mix = nn.Sequential(
            nn.Conv2d(input_channels, input_channels, kernel_size=1),
            nn.ReLU(),
        )


        layers = []
        in_ch = input_channels      # this will double on each block
        for i in range(depth):
            out_ch = base_filters * (2**i)
            layers.extend([
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2)
            ])
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_ch = out_ch


        self.cnn = nn.Sequential(*layers)                                       # combine into one module
        self.global_pool = nn.AdaptiveAvgPool2d(1)                              # global average pooling
        self.embedding = nn.Linear(out_ch, embedding_dim)                       # maps layer -> size 128
        self.classifier = nn.Linear(embedding_dim, 1)                           # maps embedding (128) -> single logic bit

    def forward(self, x):
        x = self.cnn(x)
        x = self.global_pool(x).flatten(1)      # x: (batch_size, out_ch)
        emb = self.embedding(x)                 # dense layer
        logit = self.classifier(emb)            # x: (batch_size, 1)
        return logit.squeeze(1), emb

In [ ]:
from tqdm import tqdm

def one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    """
    Train for one epoch with Mixed Precision (AMP) and Accuracy tracking.
    """
    model.train()
    losses = []
    correct = 0
    total = 0

    for x, y in tqdm(loader, desc="Training", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()

        # --- Mixed Precision Forward Pass ---
        if scaler is not None:
            with autocast():
                logits, _ = model(x)
                loss = criterion(logits, y)
            
            # --- Scaled Backwards Pass ---
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, _ = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        # Metrics
        losses.append(loss.item())
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)

    train_acc = correct / total if total > 0 else 0.0
    return np.mean(losses), train_acc


In [ ]:
@torch.inference_mode() # faster for making only predictions
def evaluate(model, loader, device):
  model.eval()

  all_labels = []
  all_preds = []
  all_probs = []

  for x, y in tqdm(loader, desc="Evaluating", leave=False):
      x, y = x.to(device), y.to(device)

      logits, _ = model(x)
      probs = torch.sigmoid(logits) # probability of fake
      preds = (probs > 0.5).long()  # threshold at 0.5

      all_labels.append(y.cpu())
      all_preds.append(preds.cpu())
      all_probs.append(probs.cpu())

  if len(all_labels) == 0:
    return 0, 0, 0  # Avoid crash for empty test set

  all_labels = torch.cat(all_labels).numpy()
  all_preds = torch.cat(all_preds).numpy()
  all_probs = torch.cat(all_probs).numpy().flatten()

  # getting metrics
  accuracy = accuracy_score(all_labels, all_preds)
  f1 = f1_score(all_labels, all_preds)

  # for AUROC, we use probabilities, not predicted classes
  try:
    auroc = roc_auc_score(all_labels, all_probs)
  except ValueError:
    auroc = float('nan') # only if on class is present

  return accuracy, f1, auroc


In [ ]:
def get_image_paths_and_labels(folder):
    """
    Return list of image paths and binary labels (0 : real, 1 : fake)
    """
    paths = []
    labels = []
    for label, subfolder in enumerate(["real", "fake"]):
        subdir = os.path.join(folder, subfolder)
        for fname in os.listdir(subdir):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                paths.append(os.path.join(subdir, fname))
                labels.append(label)
    return paths, labels

In [ ]:
# Load data

train_folder = f"{data_dir}/train"
test_folder = f"{data_dir}/test"

train_paths, train_labels = get_image_paths_and_labels(train_folder)
test_paths, test_labels = get_image_paths_and_labels(test_folder)

train_dataset = FFTDataset(train_paths, train_labels)
test_dataset = FFTDataset(test_paths, test_labels)

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size,
    shuffle=True, 
    num_workers=NUM_WORKERS, 
    pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size,
    shuffle=True, 
    num_workers=NUM_WORKERS, 
    pin_memory=PIN_MEMORY
)


In [ ]:
def find_real_sample(dataset):
    for i, lbl in enumerate(dataset.labels):
        if lbl == 0:   # assuming 0 = real
            return i
    raise ValueError("No real samples found")


def find_fake_sample(dataset):
    for i, lbl in enumerate(dataset.labels):
        if lbl == 1:   # assuming 1 = fake
            return i
    raise ValueError("No fake samples found")


def visualize_fft(dataset, index=0):
    """
    dataset: FFTDataset object
    index: which sample from dataset to visualize
    """
    # --- Load raw image path from dataset ---
    img_path = dataset.img_paths[index]
    print("Using test image:", img_path)

    # Load image manually (same as dataset code)
    img = Image.open(img_path).convert('RGB')
    resize = transforms.Resize((256, 256))
    img_resized = resize(img)

    # Step 1: RGB → Grayscale
    img_gray = transforms.functional.rgb_to_grayscale(img_resized)
    img_gray_t = transforms.functional.to_tensor(img_gray)  # (1, H, W)

    # Step 2: FFT
    fft = torch.fft.fft2(img_gray_t)

    # Step 3: FFT shift
    fft_shift = torch.fft.fftshift(fft)

    # Step 4: Magnitude
    fft_mag = torch.abs(fft_shift)

    # Step 5: Log magnitude
    fft_log_mag = torch.log1p(fft_mag)

    to_np = lambda x: x.squeeze(0).cpu().numpy()

    steps = {
        "Original (RGB)": np.array(img_resized),
        "Grayscale": to_np(img_gray_t),
        "FFT Log-Magnitude": to_np(fft_log_mag),
    }

    plt.figure(figsize=(16, 12))
    for i, (title, image) in enumerate(steps.items()):
        plt.subplot(2, 3, i+1)
        if image.ndim == 2:
            plt.imshow(image, cmap='gray')
        else:
            plt.imshow(image)
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# visualize the fft transformation

real_idx = find_real_sample(test_dataset)
fake_idx = find_fake_sample(test_dataset)

print("Real sample index:", real_idx)
visualize_fft(test_dataset, real_idx)

print("Fake sample index:", fake_idx)
visualize_fft(test_dataset, fake_idx)


In [ ]:

def main():
    # --- Training Setup ---
    best_auroc = -1.0
    best_path = ARTIFACTS_DIR / f"compact_fft_model.pth"
    history = {"train": [], "val": []}

    model = CompactFFTNet().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    scaler = GradScaler() # For Mixed Precision

    for epoch in range(epochs):
        t0 = time.time()
        dt = time.time() - t0

        tr_loss, tr_acc = one_epoch(model, train_loader, optimizer, criterion, device, scaler)
        va_acc, va_f1, va_auroc = evaluate(model, test_loader, device)

        print(f"Epoch {epoch+1}/{epochs} | {dt:.1f}s | Loss: {tr_loss:.4f} | TrAcc: {tr_acc:.4f} | ValAUROC: {va_auroc:.4f}")


        history["train"].append(tr_loss)
        history["val"].append((va_acc, va_f1, va_auroc))

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"{dt:.1f}s | "
            f"Train Loss: {tr_loss:.4f} | "
            f"Val Acc: {va_acc:.4f} | "
            f"Val F1: {va_f1:.4f} | "
            f"Val AUROC: {va_auroc:.4f}"
        )

        # Checkpointing + Early Stopping
        if not math.isnan(va_auroc) and va_auroc > best_auroc:
            best_auroc = va_auroc
            patience_counter = 0

            torch.save(
                {
                    "model_state": model.state_dict(),
                    "model_name": "compact_fft_model",
                },
                best_path,
            )

            print(f"✅ Saved new best checkpoint (val AUROC={best_auroc:.3f})")

        else:
            patience_counter += 1
            print(f"No improvement. Patience {patience_counter}/{PATIENCE}")

            if patience_counter >= PATIENCE:
                print("⏹️ Early stopping triggered!")
                break

    torch.save(model.state_dict(), "compact_fft_model.pth")
    print("Training complete. Model saved to compact_fft_model.pth")
    return history

if __name__ == "__main__":
    history = main()

In [ ]:
# Extract metrics from history
tr_loss = history["train"]  # Train loss only
va_acc = [x[0] for x in history["val"]]
va_f1 = [x[1] for x in history["val"]]
va_auroc = [x[2] for x in history["val"]]

epochs = range(1, len(tr_loss) + 1)

plt.figure(figsize=(18, 5))

# 1. Training Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, tr_loss, 'b-o', label='Train Loss')
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# 2. Validation Metrics (Accuracy, F1, AUROC)
plt.subplot(1, 2, 2)
plt.plot(epochs, va_acc, 'r-o', label='Val Acc')
plt.plot(epochs, va_f1, 'g-o', label='Val F1')
plt.plot(epochs, va_auroc, 'm-o', label='Val AUROC')
plt.title(f"Validation Metrics (Best AUROC: {max(va_auroc):.3f})")
plt.xlabel("Epoch")
plt.ylabel("Metric")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()